# Notebook 01: Data Quality & Relational Integrity Audit
**Project**: E-Commerce Discount Impact Analysis  
**Stage**: Data Ingestion, Relational Auditing, Quality Checks, and Order-Level Aggregation

In [1]:
import os
import pandas as pd
import numpy as np

from src.data_cleaning import load_raw_data, validate_data_integrity, clean_and_prepare_delivered_orders, save_processed_datasets

# 1. Load raw datasets
datasets = load_raw_data('data/raw')
for name, df in datasets.items():
    print(f"{name:15s}: {df.shape[0]:,} rows, {df.shape[1]} columns")

customers      : 1,000 rows, 7 columns
orders         : 5,000 rows, 5 columns
order_details  : 12,558 rows, 8 columns
products       : 82 rows, 5 columns
returns        : 257 rows, 4 columns


### 1. Relational Integrity and Formula Validation

In [2]:
# 2. Audit integrity
audit = validate_data_integrity(datasets)
print("--- Data Quality Audit Report ---")
for k, v in audit.items():
    print(f"  {k:30s}: {v}")

--- Data Quality Audit Report ---
  shapes                        : {'customers': (1000, 7), 'orders': (5000, 5), 'order_details': (12558, 8), 'products': (82, 5), 'returns': (257, 4)}
  null_counts                   : {'customers': {'CustomerID': 0, 'CustomerName': 0, 'Gender': 0, 'Age': 0, 'City': 0, 'SignupDate': 0, 'Region': 0}, 'orders': {'OrderID': 0, 'OrderDate': 0, 'CustomerID': 0, 'PaymentMethod': 0, 'OrderStatus': 0}, 'order_details': {'OrderDetailID': 0, 'OrderID': 0, 'ProductID': 0, 'Quantity': 0, 'UnitPrice': 0, 'DiscountPct': 0, 'SalesAmount': 0, 'Profit': 0}, 'products': {'ProductID': 0, 'ProductName': 0, 'Category': 0, 'CostPrice': 0, 'SellingPrice': 0}, 'returns': {'ReturnID': 0, 'OrderID': 0, 'ReturnDate': 0, 'ReturnReason': 0}}
  duplicate_rows                : {'customers': 0, 'orders': 0, 'order_details': 0, 'products': 0, 'returns': 0}
  orphan_order_details          : 0
  orphan_orders_customers       : 0
  orphan_order_details_products : 0
  orphan_returns      

### 2. Filter Delivered Orders and Aggregate to Order-Level Units

In [3]:
# 3. Clean and prepare delivered orders
delivered_orders, delivered_items = clean_and_prepare_delivered_orders(datasets)
save_processed_datasets(delivered_orders, delivered_items)

print(f"\nDelivered Orders (Order-Level AOV analytical unit): {len(delivered_orders):,} rows")
print(f"Delivered Line Items (Item-level transactional unit): {len(delivered_items):,} rows")
display(delivered_orders.head(3))


Delivered Orders (Order-Level AOV analytical unit): 4,386 rows
Delivered Line Items (Item-level transactional unit): 10,970 rows


,OrderID,OrderDate,CustomerID,PaymentMethod,OrderStatus,order_value,total_profit,total_quantity,item_count,avg_discount,...,Beauty,Books,Electronics,Fashion,Home,OrderYear,OrderMonth,OrderDayOfWeek,IsWeekend,CustomerTenureDays
0,O00001,2026-01-19,C0484,UPI,Delivered,49930.00,19182.00,7,2,10.0,...,1,0,1,0,0,2026,1,0,0,716
1,O00002,2026-02-14,C0450,Credit Card,Delivered,25521.25,5916.25,5,2,7.5,...,0,0,0,0,2,2026,2,5,1,1002
2,O00003,2025-11-21,C0523,Credit Card,Delivered,12541.90,4543.90,2,1,5.0,...,0,0,0,1,0,2025,11,4,0,262
